In [1]:
import microarray as ma

In [2]:
dataset = ma.datasets.read_geo_metadata("GSE19830", folder="./data")

In [3]:
import pandas as pd

fractions = []
for sample, row in zip(dataset["sample_geo_accession"], dataset["sample_source_name_ch1"], strict=True):
    fractions.append(
        [
            sample,
            *[int(i.split(" %")[0]) / 100 for i in row.split(" / ")],
        ]
    )
fractions_df = pd.DataFrame(fractions, columns=["accession", "fraction_liver", "fraction_brain", "fraction_lung"])
fractions_df.head()

,accession,fraction_liver,fraction_brain,fraction_lung
0,GSM495209,1.0,0.0,0.0
1,GSM495210,1.0,0.0,0.0
2,GSM495211,1.0,0.0,0.0
3,GSM495212,0.0,1.0,0.0
4,GSM495213,0.0,1.0,0.0


In [4]:
# Load cdf file
# !wget "https://ftp.ncbi.nlm.nih.gov/geo/platforms/GPL22nnn/GPL22499/suppl/GPL22499%5FRat2302%5FRn%5FENTREZG.cdf.gz" -O "./data/GPL22499.cdf.gz"

In [5]:
cdf_file = ma.io.parse_cdf("./data/GPL22499.cdf.gz")

In [6]:
adata_raw = ma.datasets.read_geo(
    "GSE19830",
    folder="./data",
    cdf_file=cdf_file,
    verbose=True,
)
adata_raw

[INFO] [21:27:56] Metadata folder: /home/malte/Dokumente/Github/dtangle/examples/data
[INFO] [21:27:56] Trying GSE19830 (not a file) as accession...
[INFO] [21:27:56] Skipped 0 accessions. Starting now.
[INFO] [21:27:56] Processing accession 1 of 1: 'GSE19830'
[INFO] [21:27:56] Found previous GSE file: /home/malte/Dokumente/Github/dtangle/examples/data/GSE19830_GSE.soft
[INFO] [21:27:56] Found previous GSM file: /home/malte/Dokumente/Github/dtangle/examples/data/GSE19830_GSM.soft
[INFO] [21:27:57] Found previous GSM file: /home/malte/Dokumente/Github/dtangle/examples/data/GSE19830_file_list.txt
[INFO] [21:27:57] 
Total number of processed SAMPLES files found is: 42
[INFO] [21:27:57] Total number of files after filter is: 42 
[INFO] [21:27:57] Total number of processed SERIES files found is: 0
[INFO] [21:27:57] Total number of files after filter is: 0 
[INFO] [21:27:57] Expanding metadata list...
[INFO] [21:27:57] Expanding metadata list...
[INFO] [21:27:57] File /home/malte/Dokumente/G

AnnData object with n_obs × n_vars = 42 × 377264
    obs: 'sample_name', 'cel_version', 'nrows', 'ncols', 'algorithm', 'sample_geo_accession', 'sample_name_meta', 'sample_title', 'sample_source_name_ch1', 'sample_organism_ch1', 'sample_platform_id', 'sample_series_id', 'file', 'file_size', 'file_url'
    var: 'probeset_id', 'probe_index', 'probe_type', 'gene_id', 'suffix'
    layers: 'stdevs', 'npixels', 'masks', 'outliers', 'modified'

In [8]:
ma.pp.normalize_quantile(adata_raw)
ma.pp.log2(adata_raw)

In [9]:
adata_sum = ma.pp.summarize_probesets(adata_raw)
adata_sum

/usr/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


AnnData object with n_obs × n_vars = 42 × 13934
    obs: 'sample_name', 'cel_version', 'nrows', 'ncols', 'algorithm', 'sample_geo_accession', 'sample_name_meta', 'sample_title', 'sample_source_name_ch1', 'sample_organism_ch1', 'sample_platform_id', 'sample_series_id', 'file', 'file_size', 'file_url'
    var: 'gene_id', 'n_probes', 'converged'
    uns: 'normalization', 'summarization'

In [13]:
# Merge with fractions
adata_sum.obs = adata_sum.obs.merge(
    fractions_df,
    left_on="sample_geo_accession",
    right_on="accession",
    how="left",
)
adata_sum.obs.head()

/usr/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


,sample_name,cel_version,nrows,ncols,algorithm,sample_geo_accession,sample_name_meta,sample_title,sample_source_name_ch1,sample_organism_ch1,sample_platform_id,sample_series_id,file,file_size,file_url,accession,fraction_liver,fraction_brain,fraction_lung
0,GSM495209.CEL,4,834,834,Percentile,GSM495209,nuid-0000-0094-1609,NUID-0000-0094-1609,100 % Liver / 0 % Brain / 0 % Lung,Rattus norvegicus,GPL1355,GSE19830,GSM495209.CEL.gz,2201134,ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM495n...,GSM495209,1.0,0.0,0.0
1,GSM495210.CEL,4,834,834,Percentile,GSM495210,nuid-0000-0094-1620,NUID-0000-0094-1620,100 % Liver / 0 % Brain / 0 % Lung,Rattus norvegicus,GPL1355,GSE19830,GSM495210.CEL.gz,2183951,ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM495n...,GSM495210,1.0,0.0,0.0
2,GSM495211.CEL,4,834,834,Percentile,GSM495211,nuid-0000-0094-1631,NUID-0000-0094-1631,100 % Liver / 0 % Brain / 0 % Lung,Rattus norvegicus,GPL1355,GSE19830,GSM495211.CEL.gz,2218116,ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM495n...,GSM495211,1.0,0.0,0.0
3,GSM495212.CEL,4,834,834,Percentile,GSM495212,nuid-0000-0094-1642,NUID-0000-0094-1642,0 % Liver / 100 % Brain / 0 % Lung,Rattus norvegicus,GPL1355,GSE19830,GSM495212.CEL.gz,2347159,ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM495n...,GSM495212,0.0,1.0,0.0
4,GSM495213.CEL,4,834,834,Percentile,GSM495213,nuid-0000-0094-1646,NUID-0000-0094-1646,0 % Liver / 100 % Brain / 0 % Lung,Rattus norvegicus,GPL1355,GSE19830,GSM495213.CEL.gz,2323449,ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM495n...,GSM495213,0.0,1.0,0.0


In [14]:
adata_sum.write_h5ad("./data/GSE19830_sum.h5ad")